<a href="https://colab.research.google.com/github/Sathvara-Amitkumar/RNN_Next-Word-Prediction/blob/main/next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
cd /content/drive/MyDrive/Colab Notebooks/RNN/

/content/drive/MyDrive/Colab Notebooks/RNN


In [12]:
ls

next_word_predictor.ipynb  qoute_dataset.csv


In [13]:
df = pd.read_csv('qoute_dataset.csv')

In [14]:
df.shape

(3038, 2)

In [15]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [16]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [18]:
quotes = df['quote']
quotes

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."
...,...
3033,The past beats inside me like a second heart.
3034,"Damn, Claire. Warn a guy before you do a face-..."
3035,"Can you be a girl for a few seconds?""""I'm alwa..."
3036,That's what fiction is for. It's for getting a...


In [20]:
quotes = quotes.str.lower()

In [22]:
import string
translater = str.maketrans(' ', ' ', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translater))

In [25]:
quotes.sample(5)

,quote
2728,this is a new year a new beginning and things ...
2968,dont let the muggles get you down
422,love is needing someone love is putting up wit...
1960,if you end up with a boring miserable life bec...
179,everything you can imagine is real


## Tokenizer

In [26]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [27]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [30]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:15]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10),
 ('in', 11),
 ('be', 12),
 ('not', 13),
 ('are', 14),
 ('your', 15)]

### Convert Text to Sequences

Now, let's use the fitted tokenizer to transform our text quotes into sequences of token IDs.

In [31]:
sequences = tokenizer.texts_to_sequences(quotes)
print(f"Number of sequences: {len(sequences)}")
print("Sample original quote:", quotes.iloc[0])
print("Sample tokenized sequence:", sequences[0])

Number of sequences: 3038
Sample original quote: “the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
Sample tokenized sequence: [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]


### Create N-Gram Input and Target Sequences

We will iterate through our tokenized `sequences` to build input-target pairs ($X$ and $y$) for our next-word prediction model.

In [32]:
input_sequences = []

# Loop over each sequence of token IDs to create incremental n-grams
for seq in sequences:
    for i in range(1, len(seq)):
        n_gram_sequence = seq[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total generated n-gram sequences: {len(input_sequences)}")

# To make it ready for training, we'll pad the sequences to have uniform length
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Splitting into predictors (X) and label (y)
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Total generated n-gram sequences: 85271
X shape: (85271, 745)
y shape: (85271,)
